# VTC-AUDIT — Qwen3.5-9B

This notebook reproduces the complete VTC-AUDIT inference and per-model analysis for **both benchmark manifests** used in the paper: seed 42 (primary) and seed 2026 (separately resampled replication).

**Run order.** Mount Google Drive, run all cells, and keep the notebook under `VTC_AUDIT/codes/`. Outputs are written only under:

- `VTC_AUDIT/results/run/seed_42/...`
- `VTC_AUDIT/results/run/seed_2026/...`

The protocol is fixed to 500 unique images per task, eight image shifts, DivPrune and ApET at a 25% retained-position budget, greedy deterministic decoding, and 5,000 image-cluster bootstrap replicates. Each completed run saves the inference records, case-level audit data, primary contrasts, severity summary, and monitor statistics required by notebook 04. Completed runs are resume-safe and reused only after integrity checks.


In [ ]:
%pip -q install --upgrade-strategy only-if-needed \
  "transformers==5.14.1" "datasets==4.0.0" \
  "accelerate>=1.10,<2" "huggingface_hub>=0.35,<2" \
  "sentencepiece" "safetensors" "filelock>=3.12" \
  "scikit-learn>=1.4,<2" "pyarrow>=14"

import datasets, torch, transformers

if transformers.__version__ != "5.14.1":
    raise RuntimeError(f"Expected transformers 5.14.1, got {transformers.__version__}")
if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required.")

print("torch", torch.__version__)
print("transformers", transformers.__version__)
print("datasets", datasets.__version__)
print("gpu", torch.cuda.get_device_name(0))


In [ ]:
# Fixed paper configuration
PRIMARY_SEED = 42
RESAMPLE_SEEDS = [42, 2026]
PROJECT_ROOT = "/content/drive/MyDrive/VTC_AUDIT"
PROTOCOL_TAG = "vtc_audit_wacv2027_divprune_apet_r25_uidfps_v1"

# Seed 42 is the primary paper benchmark; 2026 is the benchmark-resample robustness check.

MODEL_KIND = "qwen35"
MODEL_ID = "Qwen/Qwen3.5-9B"
MODEL_REVISION = "c202236235762e1c871ad0ccb60c8ee5ba337b9a"
MODEL_SLUG = "qwen3_5_9b"
POSITION_POLICY = "native_gapped"
TRUST_REMOTE_CODE = False

P = {"n_per_task": 500, "chunk_size": 10, "parity_n": 5, "bootstrap_n": 5000}
ACTIVE_RESAMPLE_SEEDS = list(RESAMPLE_SEEDS)

BENCHMARKS = ["pope", "textvqa", "gqa", "chartqa"]
RETAIN_RATIOS = {"r25": 0.25}
APET_BASIS_TOKENS = 10
APET_FPS_SEED = 1701
MAX_NEW_TOKENS = {"pope": 4, "textvqa": 8, "gqa": 8, "chartqa": 8}

MIN_PIXELS = 256 * 28 * 28
MAX_PIXELS = 512 * 28 * 28

DATASET_REVISIONS = {
    "HuggingFaceM4/ChartQA": "b605b6e08b57faf4359aeb2fe6a3ca595f99b6c5",
    "lmms-lab/GQA": "a6e72d6e1b912da88af8b2f9eba05d5ea8ec2dd8",
    "lmms-lab/POPE": "4db1276663dfa5eb8ad16a52d24c31a09e470896",
    "lmms-lab/textvqa": "9c0699cd19768ac5ab97568f6b3cbac4c0062884",
}

SHIFT_KEYS = [
    "jpeg_q70", "jpeg_q50", "noise_s2", "noise_s5",
    "blur_r0p5", "blur_r1", "brightness_0p9", "brightness_0p8",
]


print({"model": MODEL_ID, "primary_seed": PRIMARY_SEED, "replication_seed": 2026, "n_per_task": P["n_per_task"], "bootstrap_n": P["bootstrap_n"]})


In [ ]:
# Setup
import gc
import hashlib
import io
import json
import math
import os
import random
import re
import time
from dataclasses import dataclass, replace
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from datasets import load_dataset
from filelock import FileLock
from PIL import Image, ImageEnhance, ImageFilter
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

from google.colab import drive
drive.mount("/content/drive")

try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")

torch.set_grad_enabled(False)
torch.set_float32_matmul_precision("high")
torch.backends.cudnn.benchmark = False

ROOT = Path(PROJECT_ROOT)
RUN_ROOT = ROOT / "results" / "run"
RUN_ROOT.mkdir(parents=True, exist_ok=True)

# These globals are bound to the active seed by prepare_seed().
SEED = None
MODEL_ROOT = None
CHUNK_ROOT = None
ANALYSIS_ROOT = None
SHARED_ROOT = None
DESIGN = None
MANIFEST = None
RECORDS = None
ALL_RECORDS = None
RECORD_BY_UID = None
RESULTS = None
RESULTS_PATH = None
CASES = None

def stable_hash(value: Any, length: int = 16) -> str:
    return hashlib.sha256(str(value).encode()).hexdigest()[:length]

def stable_seed(*parts: Any) -> int:
    if SEED is None:
        raise RuntimeError("SEED is not bound.")
    value = "|".join(map(str, (SEED, *parts)))
    return int(hashlib.sha256(value.encode()).hexdigest()[:8], 16)

def jsonable(value: Any):
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (np.integer, np.floating, np.bool_)):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, dict):
        return {str(k): jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [jsonable(v) for v in value]
    raise TypeError(type(value).__name__)

def atomic_json(data: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(data, indent=2, default=jsonable), encoding="utf-8")
    os.replace(tmp, path)

def atomic_parquet(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    frame.to_parquet(tmp, index=False)
    os.replace(tmp, path)


def reseed(seed: int) -> None:
    global SEED
    SEED = int(seed)
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)


In [ ]:
# Data and scoring
SCORE_EPS = 1e-9
_PUNCT = [';', '/', '[', ']', '"', '{', '}', '(', ')', '=', '+', '\\', '_', '-', '>', '<', '@', '`', ',', '?', '!']
_PERIOD_STRIP = re.compile(r"(?!<=\d)(\.)(?!\d)")
_COMMA_STRIP = re.compile(r"(\d)(,)(\d)")
_NUMBERS = {
    "none": "0", "zero": "0", "one": "1", "two": "2", "three": "3",
    "four": "4", "five": "5", "six": "6", "seven": "7", "eight": "8",
    "nine": "9", "ten": "10",
}
_ARTICLES = {"a", "an", "the"}


def vqa_norm(text: Any) -> str:
    value = str(text).replace("\n", " ").replace("\t", " ").strip().lower()
    for punct in _PUNCT:
        if (punct + " " in value or " " + punct in value) or _COMMA_STRIP.search(value):
            value = value.replace(punct, "")
        else:
            value = value.replace(punct, " ")
    value = _PERIOD_STRIP.sub("", value)
    return " ".join(_NUMBERS.get(w, w) for w in value.split() if w not in _ARTICLES)


def short_answer(text: str) -> str:
    value = str(text).strip().split("\n")[0]
    return re.split(r"(?i)\b(?:because|therefore|so that)\b", value)[0].strip()


def score_pope(prediction: str, answer: str) -> float:
    tokens = vqa_norm(prediction).split()
    pred = tokens[0] if tokens else ""
    if pred not in {"yes", "no"}:
        pred = "yes" if "yes" in tokens else ("no" if "no" in tokens else pred)
    return float(pred == vqa_norm(answer))


def score_textvqa(prediction: str, answers: Sequence[str]) -> float:
    pred = vqa_norm(short_answer(prediction))
    gold = [vqa_norm(a) for a in answers]
    if not gold:
        return 0.0
    values = []
    for i in range(len(gold)):
        matches = sum(1 for j, answer in enumerate(gold) if j != i and answer == pred)
        values.append(min(1.0, matches / 3.0))
    return float(np.mean(values))


def score_gqa(prediction: str, answer: str) -> float:
    return float(vqa_norm(short_answer(prediction)) == vqa_norm(answer))


def score_chartqa(prediction: str, answer: str) -> float:
    pred, target = short_answer(prediction).strip(), str(answer).strip()

    def as_float(value: str):
        try:
            return float(value.rstrip("%")) / 100.0 if value.endswith("%") else float(value)
        except ValueError:
            return None

    left, right = as_float(pred), as_float(target)
    if left is not None and right is not None:
        if abs(right) <= 1e-12:
            return float(abs(left - right) <= 1e-12)
        return float(abs(left - right) / abs(right) <= 0.05)
    return float(vqa_norm(pred) == vqa_norm(target))


def score_record(bench: str, prediction: str, record: Dict[str, Any]) -> float:
    if bench == "pope":
        return score_pope(prediction, record["answer"])
    if bench == "textvqa":
        return score_textvqa(prediction, record["answers"])
    if bench == "gqa":
        return score_gqa(prediction, record["answer"])
    if bench == "chartqa":
        return score_chartqa(prediction, record["answer"])
    raise KeyError(bench)


def first_field(row: Dict[str, Any], names: Sequence[str], default=None):
    for name in names:
        if name in row and row[name] is not None:
            return row[name]
    return default


def rgb_image(value: Any) -> Image.Image:
    if isinstance(value, Image.Image):
        image = value
    elif isinstance(value, dict) and value.get("bytes") is not None:
        image = Image.open(io.BytesIO(value["bytes"]))
    elif isinstance(value, dict) and value.get("path"):
        image = Image.open(value["path"])
    elif isinstance(value, (str, os.PathLike, Path)):
        image = Image.open(value)
    elif isinstance(value, np.ndarray):
        image = Image.fromarray(value)
    else:
        raise TypeError(f"Unsupported image value: {type(value).__name__}")
    image.load()
    return image.convert("RGB")


def image_hash(image: Image.Image) -> str:
    image = image.convert("RGB")
    payload = str(image.size).encode() + image.tobytes()
    return hashlib.sha256(payload).hexdigest()[:20]


def one_answer(value: Any, bench: str) -> str:
    if isinstance(value, np.ndarray):
        value = value.tolist()
    if isinstance(value, (list, tuple)):
        if len(value) != 1:
            raise ValueError(f"{bench}: expected one answer, got {len(value)}")
        value = value[0]
    value = str(value).strip()
    if not value:
        raise ValueError(f"{bench}: empty answer")
    return value


def many_answers(value: Any, bench: str) -> List[str]:
    if isinstance(value, np.ndarray):
        value = value.tolist()
    if not isinstance(value, (list, tuple)):
        raise ValueError(f"{bench}: answers must be a list")
    values = [str(x).strip() for x in value if str(x).strip()]
    if not values:
        raise ValueError(f"{bench}: empty answer list")
    return values


def pinned_dataset(repo: str, config: Optional[str], split: str):
    kwargs = {
        "path": repo,
        "split": split,
        "revision": DATASET_REVISIONS[repo],
        "token": HF_TOKEN,
    }
    dataset = load_dataset(name=config, **kwargs) if config else load_dataset(**kwargs)
    column = "__vtc_index"
    if column in dataset.column_names:
        raise RuntimeError(f"Unexpected dataset column: {column}")
    return dataset.add_column(column, np.arange(len(dataset), dtype=np.int64))


def make_record(
    bench: str,
    image: Image.Image,
    image_id: str,
    question_id: str,
    question: str,
    source_index: int,
    image_source_index: Optional[int] = None,
    answer: Optional[str] = None,
    answers: Optional[Sequence[str]] = None,
) -> Dict[str, Any]:
    question = str(question).strip()
    image_id = str(image_id)
    question_id = str(question_id)
    uid = f"{bench}_{stable_hash(f'{bench}|{image_id}|{question_id}|{question}', 18)}"
    record = {
        "uid": uid,
        "bench": bench,
        "image_id": f"{bench}:{image_id}",
        "cluster_id": f"{bench}:{image_id}",
        "question_id": f"{bench}:{question_id}",
        "question": question,
        "image": image.convert("RGB"),
        "source_index": int(source_index),
        "image_source_index": int(source_index if image_source_index is None else image_source_index),
    }
    if answers is not None:
        record["answers"] = many_answers(answers, bench)
    elif answer is not None:
        record["answer"] = one_answer(answer, bench)
    else:
        raise ValueError("Missing answer")
    return record


def row_record(row: Dict[str, Any], bench: str) -> Dict[str, Any]:
    image = rgb_image(row["image"])
    image_id = str(first_field(row, ["image_id", "image_source", "image_path", "imgname"], ""))
    if not image_id:
        image_id = image_hash(image)
    question = str(first_field(row, ["question", "query"], ""))
    question_id = str(first_field(row, ["question_id", "id"], stable_hash(question, 18)))
    source_index = int(row["__vtc_index"])
    if bench == "textvqa":
        return make_record(
            bench, image, image_id, question_id, question, source_index,
            answers=first_field(row, ["answers"], []),
        )
    answer_names = ["label", "answer"] if bench == "chartqa" else ["answer", "label"]
    return make_record(
        bench, image, image_id, question_id, question, source_index,
        answer=first_field(row, answer_names, ""),
    )


def unique_records(dataset, bench: str, n: int) -> List[Dict[str, Any]]:
    records, seen = [], set()
    for row in dataset.shuffle(seed=SEED):
        record = row_record(row, bench)
        if record["image_id"] in seen:
            continue
        seen.add(record["image_id"])
        records.append(record)
        if len(records) == n:
            break
    if len(records) != n:
        raise RuntimeError(f"{bench}: found {len(records)} unique images, expected {n}")
    return records


def load_pope(n: int):
    return unique_records(pinned_dataset("lmms-lab/POPE", "Full", "adversarial"), "pope", n)


def load_textvqa(n: int):
    return unique_records(pinned_dataset("lmms-lab/textvqa", None, "validation"), "textvqa", n)


def load_chartqa(n: int):
    return unique_records(pinned_dataset("HuggingFaceM4/ChartQA", None, "test"), "chartqa", n)


def load_gqa(n: int):
    instructions = pinned_dataset("lmms-lab/GQA", "val_balanced_instructions", "val")
    selected, order = {}, []
    for row in instructions.shuffle(seed=SEED):
        image_id = str(row["imageId"])
        if image_id in selected:
            continue
        selected[image_id] = {
            "question_id": str(row["id"]),
            "question": str(row["question"]),
            "answer": one_answer(row["answer"], "gqa"),
            "source_index": int(row["__vtc_index"]),
        }
        order.append(image_id)
        if len(order) == n:
            break
    if len(order) != n:
        raise RuntimeError(f"gqa: found {len(order)} unique images, expected {n}")

    images = pinned_dataset("lmms-lab/GQA", "val_balanced_images", "val")
    positions = {str(value): i for i, value in enumerate(images["id"])}
    missing = set(order) - set(positions)
    if missing:
        raise RuntimeError(f"gqa image join missing {len(missing)} images")
    subset = images.select([positions[image_id] for image_id in order])

    records = []
    for image_id, row in zip(order, subset):
        item = selected[image_id]
        records.append(make_record(
            "gqa",
            rgb_image(row["image"]),
            image_id,
            item["question_id"],
            item["question"],
            item["source_index"],
            image_source_index=int(row["__vtc_index"]),
            answer=item["answer"],
        ))
    return records


LOADERS = {
    "pope": load_pope,
    "textvqa": load_textvqa,
    "gqa": load_gqa,
    "chartqa": load_chartqa,
}

In [ ]:
# Shifts and selectors
@dataclass(frozen=True)
class ShiftSpec:
    key: str
    family: str
    severity: int
    value: float


SHIFT_SPECS = [
    ShiftSpec("jpeg_q70", "jpeg", 1, 70),
    ShiftSpec("jpeg_q50", "jpeg", 2, 50),
    ShiftSpec("noise_s2", "noise", 1, 2),
    ShiftSpec("noise_s5", "noise", 2, 5),
    ShiftSpec("blur_r0p5", "blur", 1, 0.5),
    ShiftSpec("blur_r1", "blur", 2, 1.0),
    ShiftSpec("brightness_0p9", "brightness", 1, 0.9),
    ShiftSpec("brightness_0p8", "brightness", 2, 0.8),
]


def apply_shift(image: Image.Image, spec: ShiftSpec, uid: str) -> Image.Image:
    image = image.convert("RGB")
    if spec.family == "jpeg":
        buffer = io.BytesIO()
        image.save(buffer, format="JPEG", quality=int(spec.value), optimize=False)
        buffer.seek(0)
        return Image.open(buffer).convert("RGB")
    if spec.family == "noise":
        array = np.asarray(image, dtype=np.float32)
        rng = np.random.default_rng(stable_seed(uid, spec.key))
        array = np.clip(array + rng.normal(0, spec.value, array.shape), 0, 255)
        return Image.fromarray(array.astype(np.uint8))
    if spec.family == "blur":
        return image.filter(ImageFilter.GaussianBlur(float(spec.value)))
    if spec.family == "brightness":
        return ImageEnhance.Brightness(image).enhance(float(spec.value))
    raise KeyError(spec.family)



def budget_k(n_tokens: int, ratio: float) -> int:
    return max(1, min(n_tokens, int(round(n_tokens * ratio))))


def divprune_order(features: torch.Tensor, max_k: int) -> torch.Tensor:
    """Official DivPrune max-min greedy rule, adapted to a tensor-only interface.

    This is a direct transliteration of vbdi/divprune::LlavaMetaForCausalLM.DivPrune:
    cosine distance, first point with the largest nearest-neighbour distance, then
    iterative farthest-from-selected max-min updates. Selected entries are masked
    only to make degenerate duplicate-token cases well-defined.
    """
    x = F.normalize(features.float(), dim=-1, eps=1e-12)
    n = int(x.shape[0])
    max_k = max(1, min(int(max_k), n))
    distance = (1.0 - x @ x.T).float().cpu()
    order = torch.empty(max_k, dtype=torch.long)
    chosen = torch.zeros(n, dtype=torch.bool)
    for i in range(max_k):
        if i == 0:
            # Official code takes the second-smallest distance in each column;
            # the smallest is the zero diagonal entry.
            scores = torch.topk(distance, k=min(2, n), dim=0, largest=False).values[-1]
        else:
            scores = distance[order[:i]].min(dim=0).values
        scores = scores.clone()
        scores[chosen] = -float("inf")
        idx = int(torch.argmax(scores))
        order[i] = idx
        chosen[idx] = True
    return order.to(features.device)


def apet_fps_indices(features: torch.Tensor, m: int, uid: str) -> torch.Tensor:
    """Deterministic, content-record-anchored FPS basis selection for ApET.

    The released ApET FPS starts from a random token. For this causal audit we keep
    that random-start semantics while removing run-to-run noise: the local RNG seed
    is SHA256(APET_FPS_SEED | uid | N). Clean and shifted versions of one record thus
    share the same initial FPS position, while different records do not acquire a
    systematic N-bucket / spatial-position anchor. Subsequent steps are the official
    farthest-point updates.
    """
    x = features.float()
    n = int(x.shape[0])
    m = max(1, min(int(m), n))
    seed_payload = f"{APET_FPS_SEED}|{uid}|{n}".encode()
    local_seed = int(hashlib.sha256(seed_payload).hexdigest()[:8], 16)
    first = int(np.random.default_rng(local_seed).integers(0, n))
    basis = torch.empty(m, dtype=torch.long, device=x.device)
    chosen = torch.zeros(n, dtype=torch.bool, device=x.device)
    min_sqdist = torch.full((n,), float("inf"), dtype=torch.float32, device=x.device)
    idx = first
    for j in range(m):
        basis[j] = idx
        chosen[idx] = True
        sqdist = ((x - x[idx]) ** 2).sum(dim=-1)
        min_sqdist = torch.minimum(min_sqdist, sqdist)
        scores = min_sqdist.clone()
        scores[chosen] = -1.0
        if j + 1 < m:
            idx = int(torch.argmax(scores).item())
    return basis


def apet_reconstruction_errors(features: torch.Tensor, basis: torch.Tensor) -> torch.Tensor:
    """ApET Eq. (4)-(5) using the released normal-equation solve."""
    x = features.float()
    basis = basis.to(x.device).long()
    b = x[basis]
    gram = b @ b.T
    gram = gram.clone()
    gram.diagonal().add_(1e-5)  # same ridge used by the official implementation
    coeff = torch.linalg.solve(gram, b @ x.T).T
    x_hat = coeff @ b
    return torch.norm(x - x_hat, dim=-1)


def apet_select(features: torch.Tensor, k: int, uid: str) -> tuple[torch.Tensor, torch.Tensor]:
    """Paper-faithful ApET support: FPS basis + largest reconstruction errors.

    The paper explicitly states that basis tokens are retained. We therefore include
    their *actual token indices* in the support and fill the remaining budget with
    the largest-error non-basis tokens. Released LLaVA/Qwen adapters realize basis
    retention through model-specific slot injection; this common-interface version
    follows the paper's Sec. 3.3 support semantics instead of those positional hacks.
    """
    n = int(features.shape[0])
    k = max(1, min(int(k), n))
    m = min(APET_BASIS_TOKENS, k, n)
    basis = apet_fps_indices(features, m, uid)
    errors = apet_reconstruction_errors(features, basis)
    basis_mask = torch.zeros(n, dtype=torch.bool, device=features.device)
    basis_mask[basis] = True
    remaining = k - m
    if remaining:
        scores = errors.clone()
        scores[basis_mask] = -float("inf")
        extras = torch.topk(scores, k=remaining, largest=True, sorted=True).indices
        support = torch.cat([basis, extras])
    else:
        support = basis.clone()
    support = torch.sort(support).values
    if len(support) != k or len(torch.unique(support)) != k:
        raise RuntimeError("ApET support construction did not produce k unique tokens.")
    if not torch.isin(basis, support).all():
        raise RuntimeError("ApET basis tokens were not retained.")
    return support, torch.sort(basis).values


def apet_merge(features: torch.Tensor, support: torch.Tensor) -> torch.Tensor:
    """ApET similarity-based average merging for a fixed retained support.

    Every removed token is assigned to its most cosine-similar retained token and
    each resulting group is averaged, exactly matching the token-merging stage in
    the paper. Returning features in sorted support order lets the same operator be
    used in dynamic, frozen-support, and clean-feature counterfactual arms.
    """
    x = features.float()
    n, d = x.shape
    support = torch.sort(support.to(x.device).long()).values
    if len(support) == 0 or support.min() < 0 or support.max() >= n:
        raise ValueError("Invalid ApET support.")
    if len(torch.unique(support)) != len(support):
        raise ValueError("ApET support contains duplicates.")
    keep_mask = torch.zeros(n, dtype=torch.bool, device=x.device)
    keep_mask[support] = True
    retained = x[support].clone()
    removed = x[~keep_mask]
    if len(removed) == 0:
        return retained.to(features.dtype)
    r_norm = F.normalize(retained, dim=-1, eps=1e-12)
    u_norm = F.normalize(removed, dim=-1, eps=1e-12)
    assignment = torch.argmax(u_norm @ r_norm.T, dim=1)
    merged = retained.clone()
    merged.index_add_(0, assignment, removed)
    counts = torch.ones(len(support), dtype=torch.float32, device=x.device)
    counts.index_add_(0, assignment, torch.ones(len(removed), dtype=torch.float32, device=x.device))
    merged = merged / counts[:, None]
    return merged.to(features.dtype)


def compressed_visual_features(features: torch.Tensor, method: str, support: torch.Tensor):
    if method == "divprune":
        return None  # ordinary hard pruning uses the untouched retained embeddings
    if method == "apet":
        return apet_merge(features, support)
    raise KeyError(method)


def support_jaccard(left: torch.Tensor, right: torch.Tensor) -> float:
    a = set(left.detach().cpu().tolist())
    b = set(right.detach().cpu().tolist())
    return len(a & b) / max(1, len(a | b))


# Selector/compressor unit tests independent of any VLM.
_test_gen = torch.Generator().manual_seed(7)
_test_features = torch.randn(17, 8, generator=_test_gen)
_test_div = divprune_order(_test_features, 7)
assert len(_test_div) == 7 and len(torch.unique(_test_div)) == 7
_test_uid = "unit_record"
_test_apet_1, _test_basis_1 = apet_select(_test_features, 7, _test_uid)
_test_apet_2, _test_basis_2 = apet_select(_test_features, 7, _test_uid)
assert torch.equal(_test_apet_1, _test_apet_2) and torch.equal(_test_basis_1, _test_basis_2)
assert torch.isin(_test_basis_1, _test_apet_1).all()
assert apet_merge(_test_features, _test_apet_1).shape == (7, 8)
_full = torch.arange(len(_test_features))
assert torch.allclose(apet_merge(_test_features, _full), _test_features)
assert np.isclose(support_jaccard(torch.tensor([0, 1]), torch.tensor([1, 2])), 1 / 3)

# Independent regression check for the common-interface ApET operator:
# basis retention, reconstruction-error ranking, and similarity-average merging.
def _apet_reference(features: torch.Tensor, k: int, uid: str):
    x = features.float()
    n = int(x.shape[0])
    m = min(APET_BASIS_TOKENS, int(k), n)
    basis = apet_fps_indices(x, m, uid)
    b = x[basis]
    gram = b @ b.T + 1e-5 * torch.eye(m, dtype=torch.float32, device=x.device)
    coeff = torch.linalg.solve(gram, b @ x.T).T
    errors = torch.linalg.vector_norm(x - coeff @ b, dim=-1)
    basis_mask = torch.zeros(n, dtype=torch.bool, device=x.device)
    basis_mask[basis] = True
    scores = errors.clone()
    scores[basis_mask] = -float("inf")
    extras = torch.topk(scores, k=max(0, int(k) - m), largest=True).indices
    support = torch.sort(torch.cat([basis, extras])).values

    keep = torch.zeros(n, dtype=torch.bool, device=x.device)
    keep[support] = True
    retained = x[support].clone()
    removed = x[~keep]
    if len(removed):
        sim = F.cosine_similarity(removed[:, None, :], retained[None, :, :], dim=-1)
        assignment = sim.argmax(dim=1)
        merged = retained.clone()
        merged.index_add_(0, assignment, removed)
        counts = torch.ones(len(support), dtype=torch.float32, device=x.device)
        counts.index_add_(0, assignment, torch.ones(len(removed), dtype=torch.float32, device=x.device))
        retained = merged / counts[:, None]
    return support, torch.sort(basis).values, retained.to(features.dtype)

_ref_support, _ref_basis, _ref_merged = _apet_reference(_test_features, 7, _test_uid)
assert torch.equal(_test_apet_1, _ref_support)
assert torch.equal(_test_basis_1, _ref_basis)
assert torch.allclose(apet_merge(_test_features, _test_apet_1), _ref_merged, atol=1e-6, rtol=1e-6)

# UID-seeded FPS is deterministic per record without a fixed positional anchor.
_uid_starts = [int(apet_fps_indices(_test_features, 1, f"unit_{i}")[0]) for i in range(16)]
assert len(set(_uid_starts)) > 1
assert int(apet_fps_indices(_test_features, 1, "same_uid")[0]) == int(apet_fps_indices(_test_features, 1, "same_uid")[0])

print({"shifts": [spec.key for spec in SHIFT_SPECS], "compressors": ["divprune", "apet"], "retention": 0.25, "apet_regression": "passed"})


In [ ]:
# Model adapter
from transformers import AutoModelForMultimodalLM, AutoProcessor


@dataclass
class VisualContext:
    record_uid: str
    input_ids: torch.Tensor
    full_embeddings: torch.Tensor
    full_position_ids: torch.Tensor
    image_mask: torch.Tensor
    image_positions: torch.Tensor
    visual_embeds: torch.Tensor
    grid_t: int
    grid_h: int
    grid_w: int

    @property
    def n_visual_tokens(self) -> int:
        return int(self.visual_embeds.shape[0])


DTYPE = torch.bfloat16
PROCESSOR = AutoProcessor.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, token=HF_TOKEN, trust_remote_code=TRUST_REMOTE_CODE
)
if hasattr(PROCESSOR, "image_processor"):
    if hasattr(PROCESSOR.image_processor, "min_pixels"):
        PROCESSOR.image_processor.min_pixels = MIN_PIXELS
    if hasattr(PROCESSOR.image_processor, "max_pixels"):
        PROCESSOR.image_processor.max_pixels = MAX_PIXELS

MODEL = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    token=HF_TOKEN,
    dtype=DTYPE,
    device_map="auto",
    low_cpu_mem_usage=True,
    attn_implementation="sdpa",
    trust_remote_code=TRUST_REMOTE_CODE,
).eval()
TOKENIZER = PROCESSOR.tokenizer
if TOKENIZER.pad_token_id is None:
    TOKENIZER.pad_token_id = TOKENIZER.eos_token_id
DEVICE = MODEL.get_input_embeddings().weight.device
if any(str(value) in {"cpu", "disk"} for value in getattr(MODEL, "hf_device_map", {}).values()):
    raise RuntimeError("The model was offloaded to CPU or disk.")
print("model device", DEVICE)


def prompt_for(bench: str, question: str) -> str:
    suffix = "Answer only yes or no." if bench == "pope" else "Answer with a short word or phrase only. Do not explain."
    return question + "\n" + suffix


def processed_inputs(record: Dict[str, Any], image: Image.Image):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": prompt_for(record["bench"], record["question"])},
        ],
    }]
    kwargs = {"tokenize": False, "add_generation_prompt": True}
    try:
        text = PROCESSOR.apply_chat_template(messages, enable_thinking=False, **kwargs)
    except TypeError:
        text = PROCESSOR.apply_chat_template(messages, **kwargs)
    batch = PROCESSOR(text=[text], images=[image], padding=False, return_tensors="pt")
    required = {"input_ids", "attention_mask", "pixel_values", "image_grid_thw"}
    missing = required - set(batch)
    if missing:
        raise RuntimeError(f"Processor missing keys: {sorted(missing)}")
    return {key: value.to(DEVICE) if hasattr(value, "to") else value for key, value in batch.items()}


def qwen_position_ids(batch, input_ids, attention_mask):
    core = MODEL.model
    kwargs = {
        "input_ids": input_ids,
        "image_grid_thw": batch["image_grid_thw"],
        "attention_mask": attention_mask,
    }
    if "mm_token_type_ids" in batch:
        kwargs["mm_token_type_ids"] = batch["mm_token_type_ids"]
    if hasattr(core, "get_rope_index"):
        positions, rope_deltas = core.get_rope_index(**kwargs)
        core.rope_deltas = rope_deltas
    elif hasattr(core, "compute_3d_position_ids"):
        positions = core.compute_3d_position_ids(**kwargs)
    else:
        raise RuntimeError("Qwen position-id API not found.")
    if positions.shape[0] == 4:
        return positions
    if positions.shape[0] == 3:
        text_positions = attention_mask.long().cumsum(-1) - 1
        text_positions.masked_fill_(attention_mask == 0, 1)
        return torch.cat([text_positions.unsqueeze(0), positions], dim=0)
    raise RuntimeError(f"Unexpected position-id shape: {tuple(positions.shape)}")


def prepare_context(record: Dict[str, Any], image: Image.Image) -> VisualContext:
    batch = processed_inputs(record, image)
    input_ids = batch["input_ids"]
    output = MODEL.get_image_features(
        pixel_values=batch["pixel_values"],
        image_grid_thw=batch["image_grid_thw"],
        return_dict=True,
    )
    visual = output.pooler_output
    visual = visual[0] if isinstance(visual, (tuple, list)) else visual
    if visual.ndim != 2:
        visual = visual.reshape(-1, visual.shape[-1])
    deepstack = getattr(output, "deepstack_features", None)
    configured = getattr(MODEL.config.vision_config, "deepstack_visual_indexes", []) or []
    if configured or (deepstack is not None and len(deepstack)):
        raise RuntimeError("Active DeepStack bypasses post-encoder pruning.")

    image_token_id = getattr(MODEL.config, "image_token_id", None)
    if image_token_id is None:
        raise RuntimeError("image_token_id is missing.")
    image_mask = input_ids[0].eq(image_token_id)
    image_positions = torch.nonzero(image_mask, as_tuple=True)[0]
    if len(image_positions) != len(visual):
        raise RuntimeError(f"Image placeholders/features differ: {len(image_positions)} vs {len(visual)}")

    embeddings = MODEL.get_input_embeddings()(input_ids).clone()
    embeddings[0, image_positions] = visual.to(embeddings.dtype)
    position_ids = qwen_position_ids(batch, input_ids, batch["attention_mask"])

    tower = getattr(MODEL.model, "visual", None)
    merge = int(getattr(tower, "spatial_merge_size", getattr(MODEL.config.vision_config, "spatial_merge_size", 1)))
    raw_t, raw_h, raw_w = map(int, batch["image_grid_thw"][0].tolist())
    grid_h, grid_w = raw_h // merge, raw_w // merge
    if grid_h <= 0 or grid_w <= 0 or len(visual) % (grid_h * grid_w):
        raise RuntimeError("Could not infer the post-merger grid.")
    grid_t = len(visual) // (grid_h * grid_w)
    return VisualContext(
        record["uid"], input_ids, embeddings, position_ids, image_mask,
        image_positions, visual, grid_t, grid_h, grid_w,
    )


def selected_prefix(context: VisualContext, selected: torch.Tensor, selected_visual: torch.Tensor | None = None):
    selected = selected.to(context.image_positions.device).long()
    perm = torch.argsort(selected)
    selected = selected[perm]
    if len(selected) == 0 or selected.min() < 0 or selected.max() >= context.n_visual_tokens:
        raise ValueError("Invalid visual-token support.")
    if len(torch.unique(selected)) != len(selected):
        raise ValueError("Visual-token support contains duplicates.")
    full_embeddings = context.full_embeddings
    if selected_visual is not None:
        if len(selected_visual) != len(selected):
            raise ValueError("selected_visual must align one-to-one with selected support.")
        selected_visual = selected_visual.to(context.image_positions.device)[perm]
        full_embeddings = context.full_embeddings.clone()
        full_embeddings[0, context.image_positions[selected]] = selected_visual.to(full_embeddings.dtype)
    keep = ~context.image_mask.clone()
    keep[context.image_positions[selected]] = True
    embeddings = full_embeddings[:, keep]
    positions = context.full_position_ids[:, :, keep]
    return embeddings, positions

def eos_ids() -> set[int]:
    values = MODEL.generation_config.eos_token_id
    if values is None:
        values = TOKENIZER.eos_token_id
    return set(values if isinstance(values, list) else [values])


def prefix_distribution(context: VisualContext, selected: torch.Tensor, selected_visual: torch.Tensor | None = None):
    if hasattr(MODEL.model, "rope_deltas"):
        MODEL.model.rope_deltas = None
    prefix, positions = selected_prefix(context, selected, selected_visual)
    length = prefix.shape[1]
    output = MODEL(
        inputs_embeds=prefix,
        attention_mask=torch.ones((1, length), dtype=torch.long, device=DEVICE),
        position_ids=positions,
        cache_position=torch.arange(length, device=DEVICE),
        use_cache=False,
        logits_to_keep=1,
        return_dict=True,
    )
    return F.softmax(output.logits[:, -1].float(), dim=-1)[0]


def manual_generate(context: VisualContext, selected: torch.Tensor, max_new_tokens: int, selected_visual: torch.Tensor | None = None):
    if hasattr(MODEL.model, "rope_deltas"):
        MODEL.model.rope_deltas = None
    prefix, positions = selected_prefix(context, selected, selected_visual)
    prefix_length = prefix.shape[1]
    output = MODEL(
        inputs_embeds=prefix,
        attention_mask=torch.ones((1, prefix_length), dtype=torch.long, device=DEVICE),
        position_ids=positions,
        cache_position=torch.arange(prefix_length, device=DEVICE),
        use_cache=True,
        logits_to_keep=1,
        return_dict=True,
    )
    past = output.past_key_values
    generated, logprobs = [], []
    last_positions = positions[:, :, -1:].clone()
    stop = eos_ids()

    for step in range(max_new_tokens):
        logits = output.logits[:, -1].float()
        log_probs = F.log_softmax(logits, dim=-1)
        token = torch.argmax(logits, dim=-1)
        token_id = int(token.item())
        generated.append(token_id)
        logprobs.append(float(log_probs[0, token_id].item()))
        if token_id in stop:
            break
        length = prefix_length + step + 1
        next_positions = last_positions + (step + 1)
        output = MODEL(
            input_ids=token.view(1, 1),
            attention_mask=torch.ones((1, length), dtype=torch.long, device=DEVICE),
            position_ids=next_positions,
            cache_position=torch.tensor([length - 1], device=DEVICE),
            past_key_values=past,
            use_cache=True,
            logits_to_keep=1,
            return_dict=True,
        )
        past = output.past_key_values

    return {
        "prediction": TOKENIZER.decode(generated, skip_special_tokens=True).strip(),
        "mean_logprob": float(np.mean(logprobs)) if logprobs else float("nan"),
        "n_generated": len(generated),
    }


def official_generate(record: Dict[str, Any], image: Image.Image):
    batch = processed_inputs(record, image)
    output = MODEL.generate(
        **batch,
        max_new_tokens=MAX_NEW_TOKENS[record["bench"]],
        do_sample=False,
        use_cache=True,
    )
    prompt_length = batch["input_ids"].shape[1]
    return TOKENIZER.decode(output[0, prompt_length:], skip_special_tokens=True).strip()


In [ ]:
# Per-resample data binding
def prepare_seed(seed: int):
    global SEED, MODEL_ROOT, CHUNK_ROOT, ANALYSIS_ROOT, SHARED_ROOT
    global DESIGN, RECORDS, ALL_RECORDS, RECORD_BY_UID, MANIFEST

    reseed(seed)

    MODEL_ROOT = RUN_ROOT / f"seed_{SEED}" / MODEL_SLUG
    CHUNK_ROOT = MODEL_ROOT / "chunks"
    ANALYSIS_ROOT = MODEL_ROOT / "analysis"
    SHARED_ROOT = RUN_ROOT / f"seed_{SEED}" / "_shared"
    for path in [MODEL_ROOT, CHUNK_ROOT, ANALYSIS_ROOT, SHARED_ROOT]:
        path.mkdir(parents=True, exist_ok=True)

    DESIGN = {
        "status": "supplement",
        "protocol_tag": PROTOCOL_TAG,
        "primary_seed": PRIMARY_SEED,
        "resample_seed": SEED,
        "model_kind": MODEL_KIND,
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "position_policy": POSITION_POLICY,
        "greedy_decoding": True,
        "n_per_task": P["n_per_task"],
        "benchmarks": BENCHMARKS,
        "shift_keys": SHIFT_KEYS,
        "retain_ratios": RETAIN_RATIOS,
        "methods": ["divprune", "apet"],
        "matched_inference_budget_retained_fraction": 0.25,
        "compression_interface": "post_projector_visual_embeds",
        "apet_basis_tokens": APET_BASIS_TOKENS,
        "apet_basis_sampling": "fps_deterministic_uid_hashed_initial_rng",
        "apet_fps_seed_scope": "sha256(APET_FPS_SEED|uid|n_visual_tokens)",
        "apet_operator": "paper_faithful_explicit_basis_retention_plus_similarity_average_merge",
        "budget_semantics": "matched_retained_positions_and_inference_cost_not_matched_information",
        "support_jaccard_semantics": "position_support_overlap; removed ApET tokens remain represented through merging",
        "dataset_revisions": DATASET_REVISIONS,
        "transformers": transformers.__version__,
        "datasets": datasets.__version__,
    }
    design_path = MODEL_ROOT / "design.json"
    if design_path.exists():
        old = json.loads(design_path.read_text())
        if old != DESIGN:
            raise RuntimeError(f"Design mismatch at {design_path}. Use a clean run folder.")
    else:
        atomic_json(DESIGN, design_path)

    RECORDS = {bench: LOADERS[bench](P["n_per_task"]) for bench in BENCHMARKS}
    ALL_RECORDS = [record for bench in BENCHMARKS for record in RECORDS[bench]]
    RECORD_BY_UID = {record["uid"]: record for record in ALL_RECORDS}

    manifest_rows = []
    for record in ALL_RECORDS:
        answer = record.get("answers", record.get("answer"))
        manifest_rows.append({
            "uid": record["uid"],
            "bench": record["bench"],
            "image_id": record["image_id"],
            "question_id": record["question_id"],
            "question": record["question"],
            "answer_json": json.dumps(answer, ensure_ascii=False),
            "source_index": record["source_index"],
            "image_source_index": record["image_source_index"],
        })
    MANIFEST = pd.DataFrame(manifest_rows).sort_values(["bench", "uid"]).reset_index(drop=True)
    manifest_path = SHARED_ROOT / "manifest.csv"

    # All three model notebooks share the same human-readable manifest for a given resample.
    with FileLock(str(SHARED_ROOT / "manifest.lock"), timeout=600):
        if manifest_path.exists():
            locked = pd.read_csv(manifest_path, keep_default_na=False)
            locked = locked.sort_values(["bench", "uid"]).reset_index(drop=True)
            pd.testing.assert_frame_equal(locked, MANIFEST, check_dtype=False)
        else:
            MANIFEST.to_csv(manifest_path, index=False)

    assert all(vqa_norm(r["answer"]) in {"yes", "no"} for r in RECORDS["pope"])
    assert all(r["answers"] for r in RECORDS["textvqa"])
    assert all(
        score_record(r["bench"], r.get("answer", r.get("answers", [""])[0]), r) > 0
        for r in RECORDS["gqa"] + RECORDS["chartqa"]
    )

    seed_record = {
        "resample_seed": SEED,
        "manifest_rows": len(MANIFEST),
        "task_counts": MANIFEST.groupby("bench").size().to_dict(),
    }
    atomic_json(seed_record, MODEL_ROOT / "benchmark_resample.json")
    print("benchmark resample", SEED, seed_record["task_counts"])
    return seed_record


In [ ]:
# Preflight: official/manual parity and visual dependence
def distribution_tv(left: torch.Tensor, right: torch.Tensor) -> float:
    return float((0.5 * torch.abs(left - right).sum()).item())

def run_preflight():
    rows = []
    for bench in BENCHMARKS:
        for record in RECORDS[bench][:P["parity_n"]]:
            context = prepare_context(record, record["image"])
            full = torch.arange(context.n_visual_tokens, device=DEVICE)
            official = official_generate(record, record["image"])
            manual = manual_generate(context, full, MAX_NEW_TOKENS[bench])

            zero_embeddings = context.full_embeddings.clone()
            zero_embeddings[0, context.image_positions] = 0
            zero_context = replace(
                context,
                full_embeddings=zero_embeddings,
                visual_embeds=torch.zeros_like(context.visual_embeds),
            )
            visual_tv = distribution_tv(
                prefix_distribution(context, full),
                prefix_distribution(zero_context, full),
            )
            k25 = budget_k(context.n_visual_tokens, RETAIN_RATIOS["r25"])
            div_support = divprune_order(context.visual_embeds, k25).sort().values
            apet_support_1, apet_basis_1 = apet_select(context.visual_embeds, k25, record["uid"])
            apet_support_2, apet_basis_2 = apet_select(context.visual_embeds, k25, record["uid"])
            apet_visual = apet_merge(context.visual_embeds, apet_support_1)
            selector_ok = bool(
                len(div_support) == k25
                and len(torch.unique(div_support)) == k25
                and len(apet_support_1) == k25
                and len(torch.unique(apet_support_1)) == k25
                and torch.equal(apet_support_1, apet_support_2)
                and torch.equal(apet_basis_1, apet_basis_2)
                and torch.isin(apet_basis_1, apet_support_1).all()
                and tuple(apet_visual.shape) == (k25, context.visual_embeds.shape[-1])
                and torch.isfinite(apet_visual.float()).all()
            )
            rows.append({
                "seed": SEED,
                "bench": bench,
                "uid": record["uid"],
                "official": official,
                "manual": manual["prediction"],
                "official_score": score_record(bench, official, record),
                "manual_score": score_record(bench, manual["prediction"], record),
                "prediction_match": vqa_norm(official) == vqa_norm(manual["prediction"]),
                "visual_tv": visual_tv,
                "n_visual_tokens": context.n_visual_tokens,
                "grid_t": context.grid_t,
                "grid_h": context.grid_h,
                "grid_w": context.grid_w,
                "selector_ok": selector_ok,
                "divprune_k": len(div_support),
                "apet_k": len(apet_support_1),
                "apet_basis_k": len(apet_basis_1),
            })
            del context, zero_context
            gc.collect()
            torch.cuda.empty_cache()

    preflight = pd.DataFrame(rows)
    preflight.to_csv(MODEL_ROOT / "preflight.csv", index=False)
    passed = bool(
        preflight["prediction_match"].all()
        and np.allclose(preflight["official_score"], preflight["manual_score"])
        and (preflight["visual_tv"] > 1e-4).all()
        and preflight["selector_ok"].all()
    )
    report = {
        "pass": passed,
        "protocol_tag": PROTOCOL_TAG,
        "seed": SEED,
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "n": len(preflight),
        "minimum_visual_tv": float(preflight["visual_tv"].min()),
        "selector_checks_pass": bool(preflight["selector_ok"].all()),
    }
    atomic_json(report, MODEL_ROOT / "preflight.json")
    print(preflight[["bench", "prediction_match", "official_score", "manual_score", "visual_tv", "n_visual_tokens"]])
    if not passed:
        raise RuntimeError(f"Preflight failed for seed {SEED}.")
    return report


In [ ]:
# Inference
RESULT_KEY = ["bench", "uid", "method", "budget", "shift_key", "support_mode"]
EXPECTED_ROWS_PER_RECORD = 59


def result_row(
    record: Dict[str, Any],
    context: VisualContext,
    generated: Dict[str, Any],
    *,
    method: str,
    budget: str,
    shift: ShiftSpec | None,
    support_mode: str,
    selected: torch.Tensor | None,
    clean_selected: torch.Tensor | None,
) -> Dict[str, Any]:
    score = score_record(record["bench"], generated["prediction"], record)
    n = context.n_visual_tokens
    k = n if selected is None else int(len(selected))
    return {
        "seed": SEED,
        "model_kind": MODEL_KIND,
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "position_policy": POSITION_POLICY,
        "bench": record["bench"],
        "uid": record["uid"],
        "image_id": record["image_id"],
        "cluster_id": record["cluster_id"],
        "question_id": record["question_id"],
        "method": method,
        "budget": budget,
        "shift_key": "clean" if shift is None else shift.key,
        "shift_family": "clean" if shift is None else shift.family,
        "severity": 0 if shift is None else shift.severity,
        "support_mode": support_mode,
        "prediction": generated["prediction"],
        "score": float(score),
        "full_credit": bool(score >= 1.0 - SCORE_EPS),
        "zero_credit": bool(score <= SCORE_EPS),
        "mean_logprob": generated["mean_logprob"],
        "n_generated": generated["n_generated"],
        "n_visual_tokens": n,
        "k": k,
        "retained_fraction": k / max(1, n),
        "grid_t": context.grid_t,
        "grid_h": context.grid_h,
        "grid_w": context.grid_w,
        "selected": None if selected is None else json.dumps(selected.detach().cpu().tolist()),
        "clean_selected": None if clean_selected is None else json.dumps(clean_selected.detach().cpu().tolist()),
        "support_jaccard": np.nan if selected is None or clean_selected is None else support_jaccard(selected, clean_selected),
    }


def run_record(record: Dict[str, Any]) -> pd.DataFrame:
    bench = record["bench"]
    clean = prepare_context(record, record["image"])
    full = torch.arange(clean.n_visual_tokens, device=DEVICE)
    k25 = budget_k(clean.n_visual_tokens, RETAIN_RATIOS["r25"])

    clean_div = divprune_order(clean.visual_embeds, k25).sort().values
    clean_apet, _ = apet_select(clean.visual_embeds, k25, record["uid"])
    supports = {
        ("divprune", "r25"): clean_div,
        ("apet", "r25"): clean_apet,
    }
    rows = []

    generated = manual_generate(clean, full, MAX_NEW_TOKENS[bench])
    rows.append(result_row(
        record, clean, generated, method="full", budget="full", shift=None,
        support_mode="full", selected=None, clean_selected=None,
    ))

    # Clean compressed baselines at a matched 25% retained-position inference budget.
    for method in ["divprune", "apet"]:
        selected = supports[(method, "r25")]
        selected_visual = compressed_visual_features(clean.visual_embeds, method, selected)
        generated = manual_generate(clean, selected, MAX_NEW_TOKENS[bench], selected_visual)
        rows.append(result_row(
            record, clean, generated, method=method, budget="r25", shift=None,
            support_mode="clean", selected=selected, clean_selected=selected,
        ))

    for spec in SHIFT_SPECS:
        shifted = prepare_context(record, apply_shift(record["image"], spec, record["uid"]))
        clean_grid = (clean.grid_t, clean.grid_h, clean.grid_w, clean.n_visual_tokens)
        shifted_grid = (shifted.grid_t, shifted.grid_h, shifted.grid_w, shifted.n_visual_tokens)
        if shifted_grid != clean_grid:
            raise RuntimeError(f"Grid changed for {record['uid']} under {spec.key}: {clean_grid} vs {shifted_grid}")

        generated = manual_generate(shifted, full, MAX_NEW_TOKENS[bench])
        rows.append(result_row(
            record, shifted, generated, method="full", budget="full", shift=spec,
            support_mode="full", selected=None, clean_selected=None,
        ))

        shifted_supports = {
            "divprune": divprune_order(shifted.visual_embeds, k25).sort().values,
            "apet": apet_select(shifted.visual_embeds, k25, record["uid"])[0],
        }
        for method in ["divprune", "apet"]:
            clean_selected = supports[(method, "r25")]
            dynamic = shifted_supports[method]
            # C_S(X) is deterministic for a fixed support S.  For ApET this includes
            # recomputing the published similarity-average merge on the arm's feature X;
            # only the retained support is intervened on.
            for mode, context, selected in [
                ("dynamic", shifted, dynamic),
                ("frozen", shifted, clean_selected),
                ("cleanfeat_dynsupport", clean, dynamic),
            ]:
                selected_visual = compressed_visual_features(context.visual_embeds, method, selected)
                generated = manual_generate(context, selected, MAX_NEW_TOKENS[bench], selected_visual)
                rows.append(result_row(
                    record, context, generated, method=method, budget="r25", shift=spec,
                    support_mode=mode, selected=selected, clean_selected=clean_selected,
                ))
        del shifted

    del clean
    frame = pd.DataFrame(rows)
    if len(frame) != EXPECTED_ROWS_PER_RECORD:
        raise RuntimeError(f"Expected {EXPECTED_ROWS_PER_RECORD} rows, got {len(frame)} for {record['uid']}")
    if frame.duplicated(RESULT_KEY).any():
        raise RuntimeError(f"Duplicate rows for {record['uid']}")
    return frame

def valid_chunk(path: Path, expected_uids: Sequence[str]) -> bool:
    if not path.exists():
        return False
    try:
        frame = pd.read_parquet(path)
    except Exception:
        return False
    if set(frame.get("seed", [])) != {SEED}:
        return False
    if set(frame.get("model_kind", [])) != {MODEL_KIND}:
        return False
    counts = frame.groupby("uid").size().to_dict()
    expected = {uid: EXPECTED_ROWS_PER_RECORD for uid in expected_uids}
    return counts == expected and not frame.duplicated(RESULT_KEY).any()

def run_inference():
    global RESULTS, RESULTS_PATH
    for bench in BENCHMARKS:
        records = RECORDS[bench]
        bench_root = CHUNK_ROOT / bench
        bench_root.mkdir(parents=True, exist_ok=True)
        chunks = [records[i:i + P["chunk_size"]] for i in range(0, len(records), P["chunk_size"])]
        for chunk_index, chunk in enumerate(tqdm(chunks, desc=f"{MODEL_KIND}/{bench}")):
            path = bench_root / f"part_{chunk_index:04d}.parquet"
            partial_path = bench_root / f"part_{chunk_index:04d}.partial.parquet"
            expected_uids = [record["uid"] for record in chunk]
            if valid_chunk(path, expected_uids):
                partial_path.unlink(missing_ok=True)
                continue

            partial = pd.read_parquet(partial_path) if partial_path.exists() else pd.DataFrame()
            if len(partial):
                counts = partial.groupby("uid").size()
                valid_partial = (
                    set(partial.get("seed", [])) == {SEED}
                    and set(partial.get("model_kind", [])) == {MODEL_KIND}
                    and not partial.duplicated(RESULT_KEY).any()
                    and not (set(counts.index) - set(expected_uids))
                    and (counts == EXPECTED_ROWS_PER_RECORD).all()
                )
                if not valid_partial:
                    raise RuntimeError(f"Invalid partial checkpoint: {partial_path}")
            completed = set(partial["uid"]) if len(partial) else set()
            frames = [partial] if len(partial) else []
            for record in chunk:
                if record["uid"] in completed:
                    continue
                try:
                    frame = run_record(record)
                except Exception as error:
                    atomic_json({
                        "bench": bench,
                        "uid": record["uid"],
                        "type": type(error).__name__,
                        "message": str(error),
                    }, MODEL_ROOT / "last_error.json")
                    raise
                frames.append(frame)
                atomic_parquet(pd.concat(frames, ignore_index=True), partial_path)
                gc.collect()
                torch.cuda.empty_cache()
            final = pd.concat(frames, ignore_index=True)
            atomic_parquet(final, path)
            partial_path.unlink(missing_ok=True)
            if not valid_chunk(path, expected_uids):
                raise RuntimeError(f"Invalid completed chunk: {path}")

    chunk_paths = sorted(CHUNK_ROOT.glob("*/part_[0-9][0-9][0-9][0-9].parquet"))
    RESULTS = pd.concat([pd.read_parquet(path) for path in chunk_paths], ignore_index=True)
    expected_rows = len(ALL_RECORDS) * EXPECTED_ROWS_PER_RECORD
    if len(RESULTS) != expected_rows:
        raise RuntimeError(f"Expected {expected_rows} rows, found {len(RESULTS)}")
    if RESULTS.duplicated(RESULT_KEY).any():
        raise RuntimeError("Duplicate result keys found.")
    RESULTS_PATH = MODEL_ROOT / "results.parquet"
    atomic_parquet(RESULTS, RESULTS_PATH)
    print("rows", len(RESULTS), "file", RESULTS_PATH)

    return RESULTS


In [ ]:
def run_analysis():
    global CASES
    # Analysis
    def indexed(method: str, budget: str, shift_key: str, support_mode: str, bench: str) -> pd.DataFrame:
        frame = RESULTS[
            (RESULTS["method"] == method)
            & (RESULTS["budget"] == budget)
            & (RESULTS["shift_key"] == shift_key)
            & (RESULTS["support_mode"] == support_mode)
            & (RESULTS["bench"] == bench)
        ].copy()
        if frame["uid"].duplicated().any():
            raise RuntimeError("Non-unique analysis rows.")
        return frame.set_index("uid")


    def build_cases(method: str, budget: str, shifts: Sequence[ShiftSpec], counterfactual: bool) -> pd.DataFrame:
        rows = []
        for spec in shifts:
            for bench in BENCHMARKS:
                parts = {
                    "full_clean": indexed("full", "full", "clean", "full", bench),
                    "full_shift": indexed("full", "full", spec.key, "full", bench),
                    "comp_clean": indexed(method, budget, "clean", "clean", bench),
                    "dynamic": indexed(method, budget, spec.key, "dynamic", bench),
                    "frozen": indexed(method, budget, spec.key, "frozen", bench),
                }
                if counterfactual:
                    parts["counter"] = indexed(method, budget, spec.key, "cleanfeat_dynsupport", bench)
                uids = set.intersection(*(set(frame.index) for frame in parts.values()))
                if len(uids) != P["n_per_task"]:
                    raise RuntimeError(f"Incomplete case join for {method}/{budget}/{bench}/{spec.key}")
                for uid in sorted(uids):
                    full_clean = parts["full_clean"].loc[uid]
                    full_shift = parts["full_shift"].loc[uid]
                    comp_clean = parts["comp_clean"].loc[uid]
                    dynamic = parts["dynamic"].loc[uid]
                    frozen = parts["frozen"].loc[uid]
                    counter = parts.get("counter")
                    counter_row = None if counter is None else counter.loc[uid]
                    eligible = bool(full_clean.full_credit and full_shift.full_credit and comp_clean.full_credit)
                    rows.append({
                        "model_kind": MODEL_KIND,
                        "method": method,
                        "budget": budget,
                        "bench": bench,
                        "uid": uid,
                        "cluster_id": full_clean.cluster_id,
                        "shift_key": spec.key,
                        "shift_family": spec.family,
                        "severity": spec.severity,
                        "eligible": eligible,
                        "full_clean_score": float(full_clean.score),
                        "full_shift_score": float(full_shift.score),
                        "clean_comp_score": float(comp_clean.score),
                        "dynamic_score": float(dynamic.score),
                        "frozen_score": float(frozen.score),
                        "counterfactual_score": np.nan if counter_row is None else float(counter_row.score),
                        "dynamic_failure": bool(eligible and dynamic.zero_credit),
                        "frozen_failure": bool(eligible and frozen.zero_credit),
                        "counterfactual_failure": np.nan if counter_row is None else bool(eligible and counter_row.zero_credit),
                        "support_jaccard": float(dynamic.support_jaccard),
                        "support_instability": 1.0 - float(dynamic.support_jaccard),
                        "n_visual_tokens": int(dynamic.n_visual_tokens),
                        "k": int(dynamic.k),
                        "retained_fraction": float(dynamic.retained_fraction),
                        "grid_t": int(dynamic.grid_t),
                        "grid_h": int(dynamic.grid_h),
                        "grid_w": int(dynamic.grid_w),
                        "clean_mean_logprob": float(comp_clean.mean_logprob),
                        "clean_uncertainty": -float(comp_clean.mean_logprob),
                        "full_clean_prediction": full_clean.prediction,
                        "full_shift_prediction": full_shift.prediction,
                        "clean_comp_prediction": comp_clean.prediction,
                        "dynamic_prediction": dynamic.prediction,
                        "frozen_prediction": frozen.prediction,
                        "counterfactual_prediction": None if counter_row is None else counter_row.prediction,
                        "clean_selected": comp_clean.selected,
                        "dynamic_selected": dynamic.selected,
                    })
        return pd.DataFrame(rows)


    CASES = pd.concat([
        build_cases("divprune", "r25", SHIFT_SPECS, True),
        build_cases("apet", "r25", SHIFT_SPECS, True),
    ], ignore_index=True)
    atomic_parquet(CASES, ANALYSIS_ROOT / "cases.parquet")


    def cell_table(frame: pd.DataFrame, outcome: str) -> pd.DataFrame:
        rows = []
        for (bench, family), group in frame.groupby(["bench", "shift_family"], sort=True):
            eligible = group["eligible"].to_numpy(bool)
            failures = group[outcome].eq(True).to_numpy(bool)
            denominator = int(eligible.sum())
            rows.append({
                "bench": bench,
                "shift_family": family,
                "eligible": denominator,
                "failures": int((eligible & failures).sum()),
                "scfr": np.nan if denominator == 0 else float((eligible & failures).sum() / denominator),
            })
        return pd.DataFrame(rows)


    def macro_scfr(frame: pd.DataFrame, outcome: str) -> float:
        values = cell_table(frame, outcome)["scfr"].dropna()
        return float(values.mean()) if len(values) else float("nan")


    def cluster_arrays(frame: pd.DataFrame, bench: str, family: str, outcome: str, clusters: Sequence[str]):
        group = frame[(frame["bench"] == bench) & (frame["shift_family"] == family)].copy()
        eligible = group.groupby("cluster_id")["eligible"].sum().reindex(clusters, fill_value=0).to_numpy(float)
        failures = (
            group.assign(_failure=group["eligible"] & group[outcome].eq(True))
            .groupby("cluster_id")["_failure"].sum().reindex(clusters, fill_value=0).to_numpy(float)
        )
        return failures, eligible


    def bootstrap_macro(frame: pd.DataFrame, outcomes: Sequence[str], n_boot: int, seed: int):
        rng = np.random.default_rng(seed)
        clusters = {
            bench: sorted(frame.loc[frame["bench"] == bench, "cluster_id"].unique())
            for bench in BENCHMARKS
        }
        indices = {
            bench: rng.integers(0, len(values), size=(n_boot, len(values)), endpoint=False)
            for bench, values in clusters.items()
        }
        replicates = {outcome: [] for outcome in outcomes}
        for outcome in outcomes:
            cell_reps = []
            for bench in BENCHMARKS:
                for family in sorted(frame["shift_family"].unique()):
                    num, den = cluster_arrays(frame, bench, family, outcome, clusters[bench])
                    sampled_num = num[indices[bench]].sum(axis=1)
                    sampled_den = den[indices[bench]].sum(axis=1)
                    cell_reps.append(np.divide(
                        sampled_num, sampled_den,
                        out=np.full(n_boot, np.nan), where=sampled_den > 0,
                    ))
            stacked = np.stack(cell_reps, axis=1)
            count = np.isfinite(stacked).sum(axis=1)
            replicates[outcome] = np.divide(
                np.nansum(stacked, axis=1), count,
                out=np.full(n_boot, np.nan), where=count > 0,
            )
        return replicates


    def interval(values: np.ndarray):
        values = np.asarray(values, dtype=float)
        values = values[np.isfinite(values)]
        if not len(values):
            return float("nan"), float("nan")
        return tuple(np.quantile(values, [0.025, 0.975]).tolist())


    summary_rows = []
    for method, budget in [("divprune", "r25"), ("apet", "r25")]:
        frame = CASES[(CASES["method"] == method) & (CASES["budget"] == budget)].copy()
        outcomes = ["dynamic_failure", "frozen_failure"]
        if budget == "r25":
            outcomes.append("counterfactual_failure")
        reps = bootstrap_macro(frame, outcomes, P["bootstrap_n"], stable_seed("macro", method, budget))
        row = {
            "model_kind": MODEL_KIND,
            "method": method,
            "budget": budget,
            "n_configurations": len(frame),
            "n_eligible": int(frame["eligible"].sum()),
            "eligibility_coverage": float(frame["eligible"].mean()),
        }
        for outcome in outcomes:
            arm = outcome.replace("_failure", "")
            low, high = interval(reps[outcome])
            row[f"{arm}_scfr"] = macro_scfr(frame, outcome)
            row[f"{arm}_ci_low"] = low
            row[f"{arm}_ci_high"] = high
        difference = reps["dynamic_failure"] - reps["frozen_failure"]
        low, high = interval(difference)
        row["dynamic_minus_frozen"] = row["dynamic_scfr"] - row["frozen_scfr"]
        row["dynamic_minus_frozen_ci_low"] = low
        row["dynamic_minus_frozen_ci_high"] = high
        if "counterfactual_failure" in outcomes:
            persistence_den = frame["eligible"] & frame["dynamic_failure"]
            persistence_num = persistence_den & frame["counterfactual_failure"].eq(True)
            row["counterfactual_persistence"] = (
                float(persistence_num.sum() / persistence_den.sum()) if persistence_den.any() else np.nan
            )
            row["persistence_numerator"] = int(persistence_num.sum())
            row["persistence_denominator"] = int(persistence_den.sum())
        else:
            row["counterfactual_persistence"] = np.nan
            row["persistence_numerator"] = np.nan
            row["persistence_denominator"] = np.nan
        summary_rows.append(row)

    SUMMARY = pd.DataFrame(summary_rows)
    SUMMARY.to_csv(ANALYSIS_ROOT / "summary.csv", index=False)

    # Descriptive severity analysis; inferential CIs remain the aggregate CIs above.
    severity_rows = []
    for method in ["divprune", "apet"]:
        method_cases = CASES[(CASES.method == method) & (CASES.budget == "r25")].copy()
        for severity in sorted(method_cases["severity"].unique()):
            part = method_cases[method_cases["severity"] == severity].copy()
            row = {
                "model_kind": MODEL_KIND, "method": method, "budget": "r25",
                "severity": int(severity), "n_configurations": len(part),
                "n_eligible": int(part["eligible"].sum()),
                "eligibility_coverage": float(part["eligible"].mean()),
            }
            for outcome in ["dynamic_failure", "frozen_failure", "counterfactual_failure"]:
                arm = outcome.replace("_failure", "")
                row[f"{arm}_scfr"] = macro_scfr(part, outcome)
            row["dynamic_minus_frozen"] = row["dynamic_scfr"] - row["frozen_scfr"]
            severity_rows.append(row)
    SEVERITY_SUMMARY = pd.DataFrame(severity_rows)
    SEVERITY_SUMMARY.to_csv(ANALYSIS_ROOT / "severity_summary.csv", index=False)

    def safe_auc(y: np.ndarray, score: np.ndarray) -> float:
        mask = np.isfinite(score)
        y, score = y[mask].astype(int), score[mask].astype(float)
        if len(np.unique(y)) < 2:
            return float("nan")
        return float(roc_auc_score(y, score))



    def cluster_auc_bootstrap(frame: pd.DataFrame, predictor: str, n_boot: int, seed: int):
        """Cluster-bootstrap the *raw*, direction-preserving AUROC.

        Do not fold each replicate with max(AUC, 1-AUC): that operation is upward biased
        under the null. Orientation-free discrimination is used only through a conservative
        upper confidence bound derived from the raw-AUROC interval.
        """
        eligible = frame[frame["eligible"]].copy().reset_index(drop=True)
        y = eligible["dynamic_failure"].to_numpy(int)
        score = eligible[predictor].to_numpy(float)
        point_auc = safe_auc(y, score)
        if not np.isfinite(point_auc):
            return point_auc, np.full(n_boot, np.nan)
        rng = np.random.default_rng(seed)
        clusters_by_bench = {
            bench: sorted(eligible.loc[eligible.bench == bench, "cluster_id"].unique())
            for bench in BENCHMARKS
        }
        row_cluster = eligible["cluster_id"].to_numpy()
        row_bench = eligible["bench"].to_numpy()
        reps = np.full(n_boot, np.nan)
        for b in range(n_boot):
            weights = np.zeros(len(eligible), dtype=float)
            for bench, clusters in clusters_by_bench.items():
                if not clusters:
                    continue
                sampled = rng.choice(clusters, size=len(clusters), replace=True)
                counts = pd.Series(sampled).value_counts().to_dict()
                bench_mask = row_bench == bench
                weights[bench_mask] = np.array(
                    [counts.get(cluster, 0) for cluster in row_cluster[bench_mask]], dtype=float
                )
            active = weights > 0
            if len(np.unique(y[active])) == 2:
                reps[b] = roc_auc_score(y[active], score[active], sample_weight=weights[active])
        return point_auc, reps


    def orientation_free_upper_bound(raw_low: float, raw_high: float):
        """Conservative 95% upper bound on discrimination in either orientation."""
        if not (np.isfinite(raw_low) and np.isfinite(raw_high)):
            return float("nan")
        return max(float(raw_high), 1.0 - float(raw_low))


    monitor_rows = []
    for method in ["divprune", "apet"]:
        monitor_frame = CASES[(CASES.method == method) & (CASES.budget == "r25")].copy()
        for predictor in ["support_instability", "clean_uncertainty"]:
            raw_auc, reps = cluster_auc_bootstrap(
                monitor_frame, predictor, P["bootstrap_n"], stable_seed("monitor", method, predictor)
            )
            raw_low, raw_high = interval(reps)
            of_high = orientation_free_upper_bound(raw_low, raw_high)
            eligible = monitor_frame[monitor_frame.eligible].copy()
            positive_clusters = int(eligible.groupby("cluster_id")["dynamic_failure"].max().sum())
            width = raw_high - raw_low if np.isfinite(raw_low) and np.isfinite(raw_high) else np.nan
            powered = bool(positive_clusters >= 100 and np.isfinite(width) and width <= 0.15)
            monitor_rows.append({
                "model_kind": MODEL_KIND,
                "method": method,
                "budget": "r25",
                "predictor": predictor,
                "n_eligible_configurations": len(eligible),
                "n_positive_clusters": positive_clusters,
                "raw_auroc": raw_auc,
                "raw_ci_low": raw_low,
                "raw_ci_high": raw_high,
                "raw_ci_width": width,
                "orientation_free_upper_95": of_high,
                "practical_threshold": 0.65,
                "powered": powered,
                "orientation_free_ruled_out_at_0p65": bool(powered and of_high < 0.65),
                "prespecified_direction_clears_0p65": bool(powered and raw_low > 0.65),
            })
    MONITOR = pd.DataFrame(monitor_rows)
    MONITOR.to_csv(ANALYSIS_ROOT / "monitor.csv", index=False)


    output_files = [
        "results.parquet",
        "analysis/cases.parquet",
        "analysis/summary.csv",
        "analysis/severity_summary.csv",
        "analysis/monitor.csv",
    ]

    COMPLETE = {
        "pass": True,
        "protocol_tag": PROTOCOL_TAG,
        "seed": SEED,
        "model_kind": MODEL_KIND,
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
        "position_policy": POSITION_POLICY,
        "result_rows": len(RESULTS),
        "expected_result_rows": len(ALL_RECORDS) * EXPECTED_ROWS_PER_RECORD,
        "case_rows": len(CASES),
        "outputs": output_files,
    }
    atomic_json(COMPLETE, MODEL_ROOT / "complete.json")
    print(SUMMARY.round(4).to_string(index=False))
    print(SEVERITY_SUMMARY.round(4).to_string(index=False))
    print(MONITOR.round(4).to_string(index=False))
    print("complete", MODEL_ROOT / "complete.json")

    return {"summary": SUMMARY, "severity": SEVERITY_SUMMARY, "monitor": MONITOR, "complete": COMPLETE}


## Run both paper manifests

Seed 42 is the primary benchmark manifest. Seed 2026 is a separately resampled benchmark manifest used as a replication/sensitivity check. Model weights are loaded once; both runs use the same deterministic inference protocol.


In [ ]:
# Run both benchmark manifests. Existing completed runs are reused only after strict validation.
def seed_root(seed: int) -> Path:
    return RUN_ROOT / f"seed_{int(seed)}" / MODEL_SLUG

def completion_is_valid(seed: int) -> bool:
    root = seed_root(seed)
    path = root / "complete.json"
    if not path.exists():
        return False
    try:
        record = json.loads(path.read_text())
    except Exception:
        return False
    expected_rows = P["n_per_task"] * len(BENCHMARKS) * EXPECTED_ROWS_PER_RECORD
    expected_cases = P["n_per_task"] * len(BENCHMARKS) * 16
    required = [
        root / "results.parquet",
        root / "analysis" / "cases.parquet",
        root / "analysis" / "summary.csv",
        root / "analysis" / "severity_summary.csv",
        root / "analysis" / "monitor.csv",
        root / "preflight.json",
        root / "design.json",
    ]
    return bool(
        record.get("pass")
        and record.get("protocol_tag") == PROTOCOL_TAG
        and int(record.get("seed", -1)) == int(seed)
        and record.get("model_kind") == MODEL_KIND
        and int(record.get("result_rows", -1)) == expected_rows
        and int(record.get("expected_result_rows", -2)) == expected_rows
        and int(record.get("case_rows", -1)) == expected_cases
        and all(p.exists() for p in required)
    )

def cleanup_seed_state():
    global RECORDS, ALL_RECORDS, RECORD_BY_UID, MANIFEST, RESULTS, CASES
    RECORDS = None
    ALL_RECORDS = None
    RECORD_BY_UID = None
    MANIFEST = None
    RESULTS = None
    CASES = None
    gc.collect()
    torch.cuda.empty_cache()

run_records = []
for seed in ACTIVE_RESAMPLE_SEEDS:
    start = time.perf_counter()
    root = seed_root(seed)
    if completion_is_valid(seed):
        print(f"seed {seed}: validated completed run found; reusing {root}")
        record = json.loads((root / "complete.json").read_text())
        record["reused_completed_run"] = True
        run_records.append(record)
        continue

    prepare_seed(seed)
    phase = {}

    t0 = time.perf_counter()
    run_preflight()
    phase["preflight_seconds"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    run_inference()
    phase["inference_seconds"] = time.perf_counter() - t0

    t0 = time.perf_counter()
    run_analysis()
    phase["analysis_seconds"] = time.perf_counter() - t0
    phase["total_seconds"] = time.perf_counter() - start

    atomic_json({
        "protocol_tag": PROTOCOL_TAG,
        "model_kind": MODEL_KIND,
        "seed": seed,
        **phase,
    }, MODEL_ROOT / "runtime.json")

    complete_path = MODEL_ROOT / "complete.json"
    complete = json.loads(complete_path.read_text())
    complete["runtime"] = phase
    complete["reused_completed_run"] = False
    atomic_json(complete, complete_path)
    run_records.append(complete)
    cleanup_seed_state()

RUN_INDEX = pd.DataFrame([{
    "model_kind": r["model_kind"],
    "seed": int(r["seed"]),
    "result_rows": int(r["result_rows"]),
    "case_rows": int(r["case_rows"]),
    "reused_completed_run": bool(r.get("reused_completed_run", False)),
} for r in run_records])
print(RUN_INDEX.to_string(index=False))
print("Done. Run notebook 04 after all three model notebooks complete.")
